In [15]:
import logging
import os
import pandas as pd
import re
import json

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Global Configuration Variables ---
# Paths to your vocabulary CSV files
LULC_VOCAB_PATH = 'LULC.csv'  # Replace with your actual path
PROCESS_VOCAB_PATH = 'LCprocess.csv'  # Replace with your actual path
VOCAB_TERM_COLUMN = 'term'  # Column name for the terms in your CSV

# Target base labels for your EntityRuler
TARGET_BASE_LABELS = [
    "LULC", "PROCESS", "CHANGE", "SURFACE_UNIT", "COORDINATES",
    "GPE", "LOC", "DATE", "PERCENT", "QUANTITY", "CARDINAL", "RESEARCH_TERM"
]

# Base SpaCy model
BASE_SPACY_MODEL = "en_core_web_sm"

logging.info("Configuration cell executed.")

2025-07-07 17:02:49,373 - INFO - Configuration cell executed.


In [16]:
# Cell: Helper Functions (Full Version with Refined Coordinates)
# Run this cell after your Configuration Cell and before Cell 10.

import logging
import os
import pandas as pd
import re # For pattern generation and preprocess_article_text (if used later)
import json # For Doccano export (not directly used in Cell 10's NER display)
import spacy
from spacy.tokens import Doc # For Doc.has_extension, Doc.set_extension

# --- Add Doc ID extension if it doesn't exist (safe to run multiple times) ---
# This should ideally be run once when spacy is first imported.
# Placing it here ensures it's set if this cell is the first to import spacy.Doc heavily.
if not Doc.has_extension('doc_id'):
    Doc.set_extension('doc_id', default=None)
    logging.info("SpaCy Doc extension 'doc_id' registered.")

def load_vocabulary_from_csv(csv_path, term_column="term"): # Using VOCAB_TERM_COLUMN from global config
    """Loads terms from a specified column in a CSV file."""
    global VOCAB_TERM_COLUMN # Access global config if term_column not overridden
    # Use argument `term_column` if provided, else use global `VOCAB_TERM_COLUMN` if defined, else default
    effective_term_column = term_column
    if term_column == "term" and 'VOCAB_TERM_COLUMN' in globals(): # Prioritize arg, then global, then default
        effective_term_column = VOCAB_TERM_COLUMN

    logging.info(f"Attempting to load vocabulary from: {csv_path} using term column: '{effective_term_column}'")
    if not os.path.exists(csv_path):
        logging.error(f"Vocabulary file not found at path: {csv_path}")
        return set()
    try:
        try:
            df_vocab = pd.read_csv(csv_path, encoding='utf-8')
        except UnicodeDecodeError:
            logging.warning(f"UTF-8 decoding failed for {csv_path}, trying 'latin1' encoding.")
            df_vocab = pd.read_csv(csv_path, encoding='latin1')

        if effective_term_column not in df_vocab.columns:
            logging.error(f"Error: Column '{effective_term_column}' not found in vocabulary file: {csv_path}")
            logging.error(f"Available columns: {df_vocab.columns.tolist()}")
            return set()

        vocab_set = set(df_vocab[effective_term_column].dropna().astype(str).str.lower().str.strip().unique())
        vocab_set.discard('')
        logging.info(f"Successfully loaded {len(vocab_set)} unique terms from {csv_path} (column: '{effective_term_column}')")
        return vocab_set
    except Exception as e:
        logging.error(f"Error loading vocabulary from {csv_path}: {e}", exc_info=True)
        return set()

def load_data(csv_path, limit=None):
    """Loads data from CSV, combines text columns, and applies a limit."""
    # (Your existing load_data function - keeping it as you provided)
    logging.info(f"Attempting to load data from: {csv_path}")
    if not os.path.exists(csv_path):
        logging.error(f"Data file not found at path: {csv_path}")
        return None
    try:
        try:
            df = pd.read_csv(csv_path, encoding='utf-8')
        except UnicodeDecodeError:
            logging.warning(f"UTF-8 decoding failed for {csv_path}, trying 'latin1' encoding.")
            df = pd.read_csv(csv_path, encoding='latin1')

        logging.info(f"Loaded data from {csv_path}. Original shape: {df.shape}")
        text_columns_candidates = ['full_text', 'text', 'Text', 'content', 'Content', 'abstract', 'title', 'sections']
        available_columns = [col for col in text_columns_candidates if col in df.columns]

        if not available_columns:
             raise ValueError(f"Could not find suitable text columns among {text_columns_candidates}.")
        logging.info(f"Using text columns for 'full_text': {', '.join(available_columns)}")

        for col in available_columns:
             if col not in df.columns: continue # Should not happen due to check above
             df[col] = df[col].fillna('')

        # Ensure available_columns still point to existing columns after potential modifications (unlikely here)
        df['full_text'] = df[available_columns].astype(str).apply(' '.join, axis=1)
        df['full_text'] = df['full_text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
        logging.info("Combined text columns into 'full_text'.")

        if 'full_text' not in df.columns or df['full_text'].empty:
             raise ValueError("Failed to create or populate 'full_text' column or it's empty.")

        if 'doc_id' not in df.columns:
             logging.warning("No 'doc_id' column found, creating one from index.")
             df = df.reset_index().rename(columns={'index': 'doc_id'})
        else:
             df['doc_id'] = df['doc_id'].astype(str) # Ensure doc_id is string

        df = df[['doc_id', 'full_text']].copy() # Select only necessary columns
        logging.info(f"DataFrame columns after selection: {df.columns.tolist()}")

        if limit is not None and isinstance(limit, int) and limit > 0 and limit < len(df):
            logging.warning(f"Limiting data to first {limit} articles for debugging.")
            df = df.head(limit).copy()

        if df.empty or 'full_text' not in df.columns or 'doc_id' not in df.columns:
             raise ValueError("DataFrame is empty or missing essential columns ('full_text', 'doc_id') after processing and limiting.")
        logging.info(f"Processed DataFrame shape: {df.shape}")
        if not df.empty:
             logging.info(f"Example doc_id: {df['doc_id'].iloc[0]}, Example text start: '{df['full_text'].iloc[0][:200]}...'")
        return df
    except Exception as e:
        logging.error(f"Error loading/processing data CSV from {csv_path}: {e}", exc_info=True)
        return None

def _generate_patterns_for_single_vocab(nlp, vocab_set, label):
    """Generates patterns (LOWER and LEMMA) for terms in a vocabulary set."""
    # (Your existing _generate_patterns_for_single_vocab - keeping as you provided)
    vocab_patterns = []
    has_lemmatizer = nlp.has_pipe("lemmatizer")
    if not has_lemmatizer:
         logging.warning(f"SpaCy model '{nlp.meta.get('name', 'Unknown Model')}' does not appear to have a lemmatizer pipe. Will skip generating LEMMA patterns for '{label}' vocabulary.")

    for term in sorted(list(vocab_set)):
        if not term or not isinstance(term, str):
             logging.warning(f"Skipping invalid term in '{label}' vocabulary: {term}")
             continue
        try:
            doc_term = nlp(term.lower())
            if len(doc_term) == 0:
                logging.warning(f"Skipped '{label}' term '{term}': SpaCy tokenization resulted in an empty Doc.")
                continue
            lower_pattern = [{"LOWER": token.lower_} for token in doc_term if token.text.strip()]
            if lower_pattern:
                vocab_patterns.append({"label": label, "pattern": lower_pattern})
            else:
                logging.warning(f"Skipped '{label}' term '{term}': Could not generate valid LOWER patterns after tokenization/stripping.")
            if has_lemmatizer:
                 lemma_pattern = [{"LEMMA": token.lemma_} for token in doc_term if token.text.strip() and token.lemma_ and token.lemma_ != "-"]
                 if lemma_pattern:
                     vocab_patterns.append({"label": label, "pattern": lemma_pattern})
                 else:
                     logging.warning(f"Skipped '{label}' term '{term}': Could not generate valid LEMMA patterns (check tokenization/lemmatization results).")
        except Exception as e:
             logging.warning(f"Error generating patterns for '{label}' term '{term}': {e}")
    logging.info(f"Generated {len(vocab_patterns)} total patterns (LOWER + LEMMA attempts) for '{label}' vocabulary.")
    return vocab_patterns


def generate_ruler_patterns(nlp_tokenizer_like, lulc_vocab, process_vocab, extra_labels=None):
    """
    Generates a list of SpaCy patterns from vocabulary (LOWER and LEMMA sequences)
    and specific rules.
    """
    patterns = []
    global TARGET_BASE_LABELS # To access the global list defined in Config cell

    logging.info("Generating patterns from vocabularies...")
    try:
        # Ensure label is in TARGET_BASE_LABELS before generating
        if "LULC" in TARGET_BASE_LABELS and lulc_vocab:
            patterns.extend(_generate_patterns_for_single_vocab(nlp_tokenizer_like, lulc_vocab, "LULC"))
        else:
            logging.warning("Skipping LULC vocab patterns: 'LULC' not in TARGET_BASE_LABELS.")
        if "PROCESS" in TARGET_BASE_LABELS and process_vocab:
            patterns.extend(_generate_patterns_for_single_vocab(nlp_tokenizer_like, process_vocab, "PROCESS"))
        else:
            logging.warning("Skipping PROCESS vocab patterns: 'PROCESS' not in TARGET_BASE_LABELS.")
    except Exception as e:
        logging.error(f"Error generating vocabulary patterns: {e}", exc_info=True)

    # --- Specific Rule-Based Patterns ---
    if "SURFACE_UNIT" in TARGET_BASE_LABELS:
        logging.info("Generating SURFACE_UNIT patterns...")
        surface_unit_rule_patterns = [ # Copied from your example
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectare", "hectares"]}}],
            [{"TEXT": {"REGEX": r"^\d+([\.,]\d+)?$"}}, {"LOWER": {"IN": ["ha", "hectare", "hectares"]}}],
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["sq", "square"]}, "OP":"?"}, {"LOWER": {"IN": ["km", "kms", "kilometer", "kilometers", "m", "meter", "meters", "mi", "miles", "yd", "yards", "ft", "feet", "cm", "centimeter", "centimeters", "ft."]}}],
            [{"TEXT": {"REGEX": r"^\d+([\.,]\d+)?$"}}, {"LOWER": {"IN": ["sq", "square"]}, "OP":"?"}, {"LOWER": {"IN": ["km", "kms", "kilometer", "kilometers", "m", "meter", "meters", "mi", "miles", "yd", "yards", "ft", "feet", "cm", "centimeter", "centimeters", "ft."]}}],
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["acres", "acre"]}}],
            [{"TEXT": {"REGEX": r"^\d+([\.,]\d+)?$"}}, {"LOWER": {"IN": ["acres", "acre"]}}],
            [{"LIKE_NUM": True}, {"TEXT": {"REGEX": r"(km|m|mi|yd|ft|cm)[\u00B2\^2]"}}],
            [{"LOWER": {"IN": ["million", "billion", "thousand"]}, "OP": "?"}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "sqkm", "km", "m", "mi", "yd", "ft", "cm", "acres"]}}],
            [{"LOWER": {"IN": ["area", "extent", "size", "covering"]}}, {"LOWER": "of", "OP": "?"}, {"OP":"{1,4}"}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": "area"}, {"LOWER": "of", "OP":"?"}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": {"IN": ["covering", "extent"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": {"IN": ["approx.", "approximately", "around", "about"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LIKE_NUM": True}, {"TEXT": {"REGEX": r"[\u2013\u2014-]|to|and"}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": {"IN": ["between"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["and", "-"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
        ]
        for pattern in surface_unit_rule_patterns:
            if isinstance(pattern, list) and all(isinstance(p, dict) for p in pattern):
                 patterns.append({"label": "SURFACE_UNIT", "pattern": pattern})
            else: logging.warning(f"Skipping invalid SURFACE_UNIT pattern: {pattern}")
        logging.info(f"Generated {len([p for p in patterns if p.get('label') == 'SURFACE_UNIT'])} SURFACE_UNIT patterns.")
    else:
        logging.warning("Skipping SURFACE_UNIT patterns: Not in TARGET_BASE_LABELS.")


    if "COORDINATES" in TARGET_BASE_LABELS:
        logging.info("Generating COORDINATES patterns (refined)...")
        coordinate_rule_patterns = []
        num_token_regex = r"^[+-]?\d+(\.\d+)?$" # Matches numbers like 45, 45.123, -45.123

        # Pattern 1: Decimal Degrees with N/S/E/W suffix, comma separated
        coordinate_rule_patterns.append([
            {"TEXT": {"REGEX": num_token_regex}},
            {"LOWER": {"REGEX": r"^[ns]$"}, "OP": "?"},
            {"LOWER": {"REGEX": r"^(north|south|n|s\.?)$"}},
            {"TEXT": ",", "OP": "?"},
            {"TEXT": {"REGEX": num_token_regex}},
            {"LOWER": {"REGEX": r"^[ew]$"}, "OP": "?"},
            {"LOWER": {"REGEX": r"^(east|west|e|w\.?)$"}}
        ])
        # Pattern 2: Decimal Degrees with ° symbol and N/S/E/W, various separators
        degree_symbol = "°" 
        coordinate_rule_patterns.append([
            {"TEXT": {"REGEX": num_token_regex}}, {"TEXT": degree_symbol},
            {"LOWER": {"REGEX": r"^[ns]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(north|south|n|s\.?)$"}},
            {"TEXT": {"REGEX": r"^([\–\-yto]|to)$"}, "OP": "?"}, # Corrected regex for separators
            {"TEXT": {"REGEX": num_token_regex}}, {"TEXT": degree_symbol},
            {"LOWER": {"REGEX": r"^[ew]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(east|west|e|w\.?)$"}}
        ])
        # Pattern 3: Keyword prefixed coordinates
        lat_keywords = ["lat", "latitude", "lat."]; lon_keywords = ["lon", "long", "longitude", "long."]; coord_keywords = ["coordinates", "coords", "position", "location", "center", "centroid"]
        coordinate_rule_patterns.append([
            {"LOWER": {"IN": lat_keywords}}, {"TEXT": {"IN": [":", "is", "at"]}, "OP": "?"}, {"OP":"{0,2}"}, # allow few words like 'is at approx'
            {"TEXT": {"REGEX": num_token_regex}},
            {"TEXT": {"IN": [",", ";"]}, "OP": "?"}, 
            {"LOWER": {"IN": lon_keywords}, "OP": "?"}, {"TEXT": {"IN": [":", "is", "at"]}, "OP": "?"}, {"OP":"{0,2}"},
            {"TEXT": {"REGEX": num_token_regex}}
        ])
        coordinate_rule_patterns.append([
            {"LOWER": {"IN": coord_keywords}}, {"TEXT": {"IN": [":", "is", "at"]}, "OP": "?"}, {"OP":"{0,2}"},
            {"TEXT": {"REGEX": num_token_regex}},
            {"TEXT": {"IN": [",", ";", "/"]}, "OP": "?"},
            {"TEXT": {"REGEX": num_token_regex}}
        ])
        # Pattern 4: N/S/E/W prefixes for numbers (single token like N45.123)
        ns_prefix_num_regex = r"^[nsNS]([+-]?\d+(\.\d+)?)$"
        ew_prefix_num_regex = r"^[ewEW]([+-]?\d+(\.\d+)?)$"
        coordinate_rule_patterns.append([
            {"TEXT": {"REGEX": ns_prefix_num_regex}}, {"TEXT": { "IN": [",", ";"]}, "OP": "?"},
            {"TEXT": {"REGEX": ew_prefix_num_regex}}
        ])
        coordinate_rule_patterns.append([ # Order swapped
            {"TEXT": {"REGEX": ew_prefix_num_regex}}, {"TEXT": { "IN": [",", ";"]}, "OP": "?"},
            {"TEXT": {"REGEX": ns_prefix_num_regex}}
        ])
        # Pattern 5 (DMS - simplified to avoid excessive FPs, still risky)
        # Matches NUMBER symbol NUMBER symbol NUMBER symbol DIRECTION (e.g. 40 ° 20 ' 10 " N)
        # This requires the symbols to be separate tokens, which is common.
        dms_symbols_tokens = ["°", "d", "'", "m", "\"", "s" ] # common symbols as separate tokens
        coordinate_rule_patterns.append([
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"}, # Degree value and optional symbol
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"}, # Minute value and optional symbol
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"}, # Second value and optional symbol
            {"LOWER": {"REGEX": r"^[ns]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(north|south|n|s\.?)$"}}
        ])
        coordinate_rule_patterns.append([ # For Longitude DMS part
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"},
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"},
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"},
            {"LOWER": {"REGEX": r"^[ew]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(east|west|e|w\.?)$"}}
        ])
        logging.warning("DMS patterns for coordinates are approximate and might need further refinement or a dedicated parser.")
        # Pattern 6: Two LIKE_NUM tokens (broad, last resort, only if other rules don't catch)
        # Example: if preceded by "located at" or "position" (can be made more specific)
        coordinate_rule_patterns.append([
            {"LOWER": {"IN": ["at", "is", "point"]}, "OP": "?"}, # context keyword
            {"LIKE_NUM": True},
            {"TEXT": {"IN": [",", ";", "/", "-"]}, "OP": "?"},
            {"LIKE_NUM": True}
        ])
        logging.warning("General paired LIKE_NUM patterns for coordinates are broad; context is key.")

        num_coord_patterns_added = 0
        for pattern_definition in coordinate_rule_patterns:
            if isinstance(pattern_definition, list) and len(pattern_definition) > 0 and \
               all(isinstance(token_matcher, dict) for token_matcher in pattern_definition):
                patterns.append({"label": "COORDINATES", "pattern": pattern_definition})
                num_coord_patterns_added +=1
            else:
                logging.warning(f"Skipping invalid COORDINATES pattern structure: {pattern_definition}")
        logging.info(f"Generated {num_coord_patterns_added} COORDINATES patterns from rules.")
    else:
        logging.warning("Skipping COORDINATES patterns: Not in TARGET_BASE_LABELS.")

    if "CHANGE" in TARGET_BASE_LABELS:
        logging.info("Generating CHANGE patterns...")
        change_lemma_terms = ["increase", "decrease", "loss", "gain", "expansion", "expand", "reduction", "grow", "growth", "change", "decline", "improve", "transform", "vary", "fluctuate", "shift", "transition", "convert", "alter", "mutation", "modify", "impact", "effect", "cause", "drive", "shift", "driver"]
        change_rule_patterns_list = [] # Use a temporary list for this section's patterns

        has_lemmatizer_for_change = nlp_tokenizer_like.has_pipe("lemmatizer")
        if has_lemmatizer_for_change:
             change_rule_patterns_list.append({"label": "CHANGE", "pattern": [{"LEMMA": {"IN": change_lemma_terms}}]})
             logging.info("Added CHANGE lemma pattern (lemmatizer detected).")
        else:
             logging.warning(f"Skipping CHANGE lemma pattern: SpaCy model lacks a lemmatizer pipe.")

        # Add specific multi-token phrase patterns (your original ones)
        change_rule_patterns_list.extend([
             {"label": "CHANGE", "pattern": [{"LOWER": {"IN": ["as", "of", "in", "to", "for"]}}, {"LOWER": "change"}]},
             # Add more multi-token patterns for CHANGE here if you have them
        ])
        for pattern_dict in change_rule_patterns_list:
            patterns.append(pattern_dict) # Append valid dict directly
        logging.info(f"Generated {len([p for p in patterns if p.get('label') == 'CHANGE'])} CHANGE patterns.")
    else:
        logging.warning("Skipping CHANGE patterns: Not in TARGET_BASE_LABELS.")

    if "RESEARCH_TERM" in TARGET_BASE_LABELS:
        candidate_label = "RESEARCH_TERM" # Your original label
        logging.info(f"Adding patterns for '{candidate_label}' (candidate)...")
        # Using the structure from your file for RESEARCH_TERM
        candidate_patterns_list = [
            {"label": candidate_label, "pattern": [{"LOWER": {"IN": ["candidate", "candidates"]}}]},
            {"label": candidate_label, "pattern": [{"LOWER": {"IN": ["candidate", "candidates"]}}, {"LOWER": {"IN": ["area", "areas", "site", "sites", "location", "locations"]}}]},
            {"label": candidate_label, "pattern": [{"LOWER": {"IN": ["candidate", "candidates"]}}, {"LOWER": "for"}, {"OP":"+"}]},
        ]
        for pattern_dict in candidate_patterns_list:
             patterns.append(pattern_dict)
        logging.info(f"Added {len([p for p in patterns if p.get('label') == candidate_label])} '{candidate_label}' related patterns.")
    else:
         logging.warning(f"Skipping patterns for 'RESEARCH_TERM': Not in TARGET_BASE_LABELS.")

    if extra_labels: # Assuming extra_labels is a dict: {'LABEL_NAME': [pattern_list_for_label_1, ...]}
        logging.info(f"Processing extra patterns from extra_labels...")
        for label, label_patterns_list in extra_labels.items():
             if label not in TARGET_BASE_LABELS: # Assuming TARGET_BASE_LABELS is globally defined
                  logging.warning(f"Skipping extra patterns for label '{label}': Not in TARGET_BASE_LABELS.")
                  continue
             logging.info(f"Adding {len(label_patterns_list)} patterns for label '{label}'.")
             for pattern_list_content in label_patterns_list: # label_patterns_list contains the actual spaCy patterns
                 if isinstance(pattern_list_content, list) and len(pattern_list_content) > 0 and all(isinstance(p_token, dict) for p_token in pattern_list_content):
                      patterns.append({"label": label, "pattern": pattern_list_content})
                 else:
                      logging.warning(f"Skipping invalid extra pattern for label '{label}': {pattern_list_content}")
    else:
        logging.info("No extra patterns provided.")
    logging.info(f"Generated a total of {len(patterns)} patterns for EntityRuler.")
    return patterns

# --- SpaCy Pipeline Setup Functions (from your file) ---
def add_ruler_to_nlp_pipeline(nlp, patterns, ruler_name="entity_ruler"):
    # (Your existing add_ruler_to_nlp_pipeline function - keeping as you provided)
    if ruler_name in nlp.pipe_names:
         logging.info(f"Removing existing '{ruler_name}' pipe for ruler-only report setup.")
         nlp.remove_pipe(ruler_name)

    # Ensure TARGET_BASE_LABELS is accessible
    global TARGET_BASE_LABELS
    if 'TARGET_BASE_LABELS' not in globals():
        logging.error("FATAL: TARGET_BASE_LABELS not defined globally. Cannot filter patterns for ruler.")
        # Handle this error, perhaps by setting TARGET_BASE_LABELS to a default list of all unique labels in patterns
        # For now, let's assume it's defined from the Config cell.
        # return # or raise an error

    report_patterns = [p for p in patterns if p.get('label') in TARGET_BASE_LABELS]
    logging.info(f"Adding {len(report_patterns)} patterns to the ruler-only report pipeline (filtered by TARGET_BASE_LABELS).")

    if not report_patterns:
         logging.warning("No relevant patterns available for ruler-only report pipeline after filtering.")
         return
    try:
        ruler = nlp.add_pipe("entity_ruler", name=ruler_name, config={"overwrite_ents": True}, last=True)
        logging.info(f"Added new '{ruler_name}' pipe using add_pipe at the end.")
        ruler.add_patterns(report_patterns)
        logging.info(f"Standard SpaCy EntityRuler '{ruler_name}' configured with {len(report_patterns)} patterns.")
    except Exception as e:
        logging.error(f"Failed to add patterns to standard SpaCy EntityRuler '{ruler_name}': {e}", exc_info=True);
        raise e

def setup_nlp_pipeline_for_doccano(base_model_name, patterns_to_add, target_labels_for_ruler_rules):
    """Sets up a SpaCy NLP pipeline with a base model, its default 'ner' (if present),
    and an EntityRuler configured with the provided patterns for custom entities."""
    # (Your existing setup_nlp_pipeline_for_doccano function - keeping as you provided)
    nlp_for_doccano = None
    try:
        logging.info(f"Loading base SpaCy model '{base_model_name}' for Doccano processing (including default NER).")
        nlp_for_doccano = spacy.load(base_model_name)
        logging.info(f"SpaCy model '{base_model_name}' loaded. Default pipes: {nlp_for_doccano.pipe_names}")

        ruler_custom_patterns_filtered = [p for p in patterns_to_add if p.get('label') in target_labels_for_ruler_rules]
        if not ruler_custom_patterns_filtered:
            logging.warning(f"No patterns relevant to your custom target labels {target_labels_for_ruler_rules} found for the EntityRuler.")
        else:
            logging.info(f"Found {len(ruler_custom_patterns_filtered)} patterns for your custom EntityRuler.")

        ruler_pipe_name = "custom_entity_ruler"
        if ruler_pipe_name in nlp_for_doccano.pipe_names:
            logging.info(f"Removing existing '{ruler_pipe_name}' before adding a new one.")
            nlp_for_doccano.remove_pipe(ruler_pipe_name)

        # Determine position for the ruler
        ruler_config = {"overwrite_ents": True}
        if "ner" in nlp_for_doccano.pipe_names:
            logging.info(f"Adding EntityRuler '{ruler_pipe_name}' before default 'ner' pipe.")
            ruler = nlp_for_doccano.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, before="ner")
        else:
            logging.info(f"Adding EntityRuler '{ruler_pipe_name}' (default 'ner' pipe not found).")
            ruler = nlp_for_doccano.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, last=True)
        
        if ruler_custom_patterns_filtered:
            ruler.add_patterns(ruler_custom_patterns_filtered)
            logging.info(f"Custom EntityRuler '{ruler_pipe_name}' configured with {len(ruler_custom_patterns_filtered)} patterns.")
        else:
            logging.info(f"Custom EntityRuler '{ruler_pipe_name}' added with 0 relevant patterns.")
            
        logging.info(f"Final pipeline for Doccano processing: {nlp_for_doccano.pipe_names}")
        return nlp_for_doccano
    except Exception as e:
        logging.error(f"FATAL: Failed to set up NLP pipeline for Doccano: {e}", exc_info=True)
        if nlp_for_doccano: del nlp_for_doccano # Clean up if part of it was loaded
        return None

    def normalize_label(label): # Nested helper, ensure it's accessible or defined globally if used elsewhere
        return 'LOC' if label in ['GPE', 'NORP'] else label

    annotations = []
    for ent in doc.ents:
        label = normalize_label(ent.label_) # Use the nested or global normalize_label
        if target_labels and label not in target_labels: # Filter by target_labels if provided
            continue
        annotations.append([ent.start_char, ent.end_char, label])

    doccano_entry = {"text": original_text, "labels": annotations }
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    file_name = f"{doc_id}.json" if doc_id else f"doc_{hash(original_text)}.json"
    file_path = os.path.join(output_dir, file_name)
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(doccano_entry, f, ensure_ascii=False, indent=2)
        logging.info(f"Saved: {file_path}")
    except Exception as e:
        logging.error(f"Error saving file '{file_path}': {e}", exc_info=True)

logging.info("Helper functions (Data Loading, Ruler/Pattern Generation, SpaCy Pipeline Setup, Doccano Export) cell defined/re-defined.")

2025-07-07 17:07:11,626 - INFO - Helper functions (Data Loading, Ruler/Pattern Generation, SpaCy Pipeline Setup, Doccano Export) cell defined/re-defined.


In [17]:
# Cell 10: NER on Sentences Loaded from CSV (Saving as JSON)

import spacy
from spacy.tokens import Doc
import pandas as pd
import logging
import json # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< IMPORT JSON MODULE

# --- Configuration for this Cell ---
SENTENCES_CSV_PATH = "all_segmented_sentences_from_articles.csv"
ARTICLE_ID_COL_IN_SENT_CSV = 'article_id'
SENTENCE_TEXT_COL_IN_SENT_CSV = 'sentence'
# This will be the name of your output JSON file
OUTPUT_NER_JSON_PATH = "extracted_ALL_entities_structured2.json" # <<<<<<<<<< DEFINE OUTPUT JSON PATH

# Ensure TARGET_BASE_LABELS, LULC_VOCAB_PATH, PROCESS_VOCAB_PATH, BASE_SPACY_MODEL are defined globally

# Helper function for label normalization
def normalize_label(label_str):
    if label_str in ['GPE', 'NORP']:
        return 'LOC'
    return label_str

# --- 0. Load LULC-Related Sentences from CSV ---
sentences_to_process_df = pd.DataFrame()
try:
    logging.info(f"Loading LULC-related sentences from: {SENTENCES_CSV_PATH}")
    sentences_to_process_df = pd.read_csv(SENTENCES_CSV_PATH)
    if ARTICLE_ID_COL_IN_SENT_CSV not in sentences_to_process_df.columns or \
       SENTENCE_TEXT_COL_IN_SENT_CSV not in sentences_to_process_df.columns:
        raise ValueError(f"CSV must contain columns '{ARTICLE_ID_COL_IN_SENT_CSV}' and '{SENTENCE_TEXT_COL_IN_SENT_CSV}'")
    logging.info(f"Successfully loaded {len(sentences_to_process_df)} sentences from {SENTENCES_CSV_PATH}.")
except FileNotFoundError:
    logging.error(f"ERROR: Sentences CSV file not found at {SENTENCES_CSV_PATH}.")
except ValueError as ve:
    logging.error(f"ERROR: {ve}")
except Exception as e:
    logging.error(f"An unexpected error occurred loading sentences CSV: {e}")

# --- 1. Load Vocabularies ---
lulc_vocab = load_vocabulary_from_csv(LULC_VOCAB_PATH) if 'LULC_VOCAB_PATH' in globals() and LULC_VOCAB_PATH else set()
process_vocab = load_vocabulary_from_csv(PROCESS_VOCAB_PATH) if 'PROCESS_VOCAB_PATH' in globals() and PROCESS_VOCAB_PATH else set()

# --- 2. Setup SpaCy NLP Pipeline ---
nlp_ner_pipeline = None
if 'generate_ruler_patterns' in globals() and 'BASE_SPACY_MODEL' in globals():
    try:
        logging.info(f"Loading base SpaCy model '{BASE_SPACY_MODEL}' for NER...")
        nlp_ner_pipeline = spacy.load(BASE_SPACY_MODEL)
        # ( ... rest of your pipeline setup from the previous full cell version ... )
        # This includes loading vocabs, generating patterns, adding entity ruler
        logging.info(f"SpaCy model '{BASE_SPACY_MODEL}' loaded. Default pipes: {nlp_ner_pipeline.pipe_names}")
        logging.info("Generating NER patterns...")
        all_patterns_for_ruler = generate_ruler_patterns(nlp_ner_pipeline, lulc_vocab, process_vocab)

        ruler_pipe_name = "custom_entity_ruler"
        if ruler_pipe_name in nlp_ner_pipeline.pipe_names:
            nlp_ner_pipeline.remove_pipe(ruler_pipe_name)
        
        ruler_config = {"overwrite_ents": True}
        if "ner" in nlp_ner_pipeline.pipe_names:
            ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, before="ner")
        else:
            ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, last=True)
        
        if all_patterns_for_ruler:
            ruler.add_patterns(all_patterns_for_ruler)
            logging.info(f"Custom EntityRuler '{ruler_pipe_name}' configured with {len(all_patterns_for_ruler)} patterns.")
        else:
            logging.warning(f"Custom EntityRuler '{ruler_pipe_name}' added, but no patterns generated.")
        logging.info(f"Final NER pipeline: {nlp_ner_pipeline.pipe_names}")

    except Exception as e:
        logging.error(f"FATAL: Failed to set up NLP pipeline for NER: {e}", exc_info=True)
        nlp_ner_pipeline = None
else:
    logging.error("Required globals 'generate_ruler_patterns' or 'BASE_SPACY_MODEL' not found.")
    nlp_ner_pipeline = None

# --- 3. Process Sentences and Extract Entities ---
extracted_entities_data_structured = [] # Renamed for clarity
if nlp_ner_pipeline and not sentences_to_process_df.empty:
    print(f"\n--- Extracting Entities from {len(sentences_to_process_df)} LULC-related Sentences (from CSV) ---")
    for index, row in sentences_to_process_df.iterrows():
        article_id = row.get(ARTICLE_ID_COL_IN_SENT_CSV, f"Row_{index}")
        sentence_text = row.get(SENTENCE_TEXT_COL_IN_SENT_CSV, "")

        if not isinstance(sentence_text, str) or not sentence_text.strip():
            continue

        doc = nlp_ner_pipeline(sentence_text)
        entities_in_sentence = []
        if doc.ents:
            for ent in doc.ents:
                normalized_ent_label = normalize_label(ent.label_)
                if 'TARGET_BASE_LABELS' in globals() and normalized_ent_label in TARGET_BASE_LABELS:
                    entities_in_sentence.append({
                        'text': ent.text,
                        'label': normalized_ent_label,
                        'start_char': ent.start_char,
                        'end_char': ent.end_char
                    })
        
        # Append entry even if no entities are found for that sentence, if you want to keep all sentences
        extracted_entities_data_structured.append({
             'article_id': article_id,
             'original_sentence': sentence_text,
             'entities': entities_in_sentence # This will be an empty list if no entities found
         })
    print("\n--- Finished Entity Extraction ---")
elif sentences_to_process_df.empty:
    print("Skipping entity extraction: No sentences loaded from the CSV file.")
elif not nlp_ner_pipeline:
    print("Skipping entity extraction: NER pipeline not set up.")

# --- 4. Display Sample Results and Save as JSON File ---
if extracted_entities_data_structured:
    print(f"\n--- Extracted Entities Data (showing first 3 entries) ---")
    for i, entry in enumerate(extracted_entities_data_structured[:3]):
        print(f"\nArticle ID: {entry['article_id']}")
        print(f"Sentence: {entry['original_sentence']}")
        print("Entities:")
        if entry['entities']:
            for entity in entry['entities']:
                print(f"  - Text: '{entity['text']}', Label: {entity['label']}, Start: {entity['start_char']}, End: {entity['end_char']}")
        else:
            print("  (No entities extracted for this sentence based on TARGET_BASE_LABELS)")
    
    # --- SAVE AS A SINGLE JSON FILE ---
    try:
        with open(OUTPUT_NER_JSON_PATH, 'w', encoding='utf-8') as f_json:
            json.dump(extracted_entities_data_structured, f_json, ensure_ascii=False, indent=4) # indent for readability
        print(f"\n--- Successfully saved structured NER data to: {OUTPUT_NER_JSON_PATH} (single JSON file) ---")
    except Exception as e_json_save:
        print(f"\nError saving structured NER data to JSON: {e_json_save}")

else:
    print("\nNo data from entity extraction to display or save.")

2025-07-07 17:13:25,353 - INFO - Loading LULC-related sentences from: all_segmented_sentences_from_articles.csv
2025-07-07 17:13:25,395 - INFO - Successfully loaded 13660 sentences from all_segmented_sentences_from_articles.csv.
2025-07-07 17:13:25,397 - INFO - Attempting to load vocabulary from: LULC.csv using term column: 'term'
2025-07-07 17:13:25,402 - INFO - Successfully loaded 172 unique terms from LULC.csv (column: 'term')
2025-07-07 17:13:25,403 - INFO - Attempting to load vocabulary from: LCprocess.csv using term column: 'term'
2025-07-07 17:13:25,406 - INFO - Successfully loaded 19 unique terms from LCprocess.csv (column: 'term')
2025-07-07 17:13:25,408 - INFO - Loading base SpaCy model 'en_core_web_sm' for NER...
2025-07-07 17:13:25,817 - INFO - SpaCy model 'en_core_web_sm' loaded. Default pipes: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
2025-07-07 17:13:25,819 - INFO - Generating NER patterns...
2025-07-07 17:13:25,820 - INFO - Generating patte


--- Extracting Entities from 13660 LULC-related Sentences (from CSV) ---

--- Finished Entity Extraction ---

--- Extracted Entities Data (showing first 3 entries) ---

Article ID: Article_1
Sentence: Land use and land cover change detection and prediction in Bhutan's high altitude city of Thimphu, using cellular automata and Markov chain Rapid urbanization is changing landscapes often resulting in the degradation of ecosystem services and quality of urban life.
Entities:
  - Text: 'change', Label: CHANGE, Start: 24, End: 30
  - Text: 'Bhutan', Label: LOC, Start: 59, End: 65
  - Text: 'city', Label: LULC, Start: 82, End: 86
  - Text: 'Thimphu', Label: LOC, Start: 90, End: 97
  - Text: 'urbanization', Label: PROCESS, Start: 146, End: 158
  - Text: 'changing', Label: CHANGE, Start: 162, End: 170
  - Text: 'urban', Label: LULC, Start: 254, End: 259

Article ID: Article_1
Sentence: Remote sensing and GIS tools can provide valuable information to deepen our understanding of the dynamics of

In [5]:
# Test Cell: Run this AFTER running your Configuration and Helper Functions cells

import pandas as pd
import os
import json

print("=== TESTING YOUR ACTUAL CODE ===\n")

# Step 1: Create test vocabulary CSV files with multi-word terms
print("1. Creating test vocabulary files...")

# Create test LULC vocabulary
lulc_test_data = pd.DataFrame({
    'term': [
        'forest',                    # single word
        'tropical forest',           # multi-word
        'agricultural land',         # multi-word
        'urban area',                # multi-word
        'primary forest',            # multi-word
        'grassland',                 # single word
        'built-up area',             # multi-word
        'bare soil',                 # multi-word
        'mixed forest',              # multi-word
        'wetland'                    # single word
    ]
})

# Create test Process vocabulary
process_test_data = pd.DataFrame({
    'term': [
        'deforestation',             # single word
        'land degradation',          # multi-word
        'urban expansion',           # multi-word
        'forest fragmentation',      # multi-word
        'soil erosion',              # multi-word
        'habitat loss',              # multi-word
        'land use change',           # three words!
        'agricultural intensification', # multi-word
        'conversion',                # single word
        'natural regeneration'       # multi-word
    ]
})

# Save test vocabularies
lulc_test_data.to_csv('test_LULC.csv', index=False)
process_test_data.to_csv('test_LCprocess.csv', index=False)

# Step 2: Create test sentences CSV
test_sentences_data = pd.DataFrame({
    'article_id': ['test_001', 'test_002', 'test_003', 'test_004', 'test_005'],
    'sentence': [
        'The tropical forest experienced severe land degradation due to urban expansion.',
        'Agricultural land was converted through forest fragmentation and soil erosion.',
        'Primary forest showed natural regeneration after agricultural intensification stopped.',
        'Land use change from mixed forest to built-up area affected 200 hectares.',
        'Bare soil increased due to deforestation and habitat loss in the wetland.'
    ]
})

test_sentences_data.to_csv('test_sentences.csv', index=False)

print("   ✓ Created test_LULC.csv")
print("   ✓ Created test_LCprocess.csv")
print("   ✓ Created test_sentences.csv")

# Step 3: Update your configuration variables
print("\n2. Updating configuration variables...")
LULC_VOCAB_PATH = 'test_LULC.csv'
PROCESS_VOCAB_PATH = 'test_LCprocess.csv'
SENTENCES_CSV_PATH = 'test_sentences.csv'
OUTPUT_NER_JSON_PATH = 'test_output_entities.json'

print(f"   LULC_VOCAB_PATH = '{LULC_VOCAB_PATH}'")
print(f"   PROCESS_VOCAB_PATH = '{PROCESS_VOCAB_PATH}'")
print(f"   SENTENCES_CSV_PATH = '{SENTENCES_CSV_PATH}'")

# Step 4: Now run YOUR Cell 10 code with test data
print("\n3. Running YOUR entity extraction code...")
print("="*60)

# ========== COPY YOUR CELL 10 CODE HERE ==========
# This is YOUR code from Cell 10, just with the test paths

import spacy
from spacy.tokens import Doc
import pandas as pd
import logging
import json

# Configuration (using test paths)
ARTICLE_ID_COL_IN_SENT_CSV = 'article_id'
SENTENCE_TEXT_COL_IN_SENT_CSV = 'sentence'

# Your normalize_label function
def normalize_label(label_str):
    if label_str in ['GPE', 'NORP']:
        return 'LOC'
    return label_str

# --- YOUR CODE STARTS HERE ---
# 0. Load sentences
sentences_to_process_df = pd.DataFrame()
try:
    logging.info(f"Loading LULC-related sentences from: {SENTENCES_CSV_PATH}")
    sentences_to_process_df = pd.read_csv(SENTENCES_CSV_PATH)
    if ARTICLE_ID_COL_IN_SENT_CSV not in sentences_to_process_df.columns or \
       SENTENCE_TEXT_COL_IN_SENT_CSV not in sentences_to_process_df.columns:
        raise ValueError(f"CSV must contain columns '{ARTICLE_ID_COL_IN_SENT_CSV}' and '{SENTENCE_TEXT_COL_IN_SENT_CSV}'")
    logging.info(f"Successfully loaded {len(sentences_to_process_df)} sentences from {SENTENCES_CSV_PATH}.")
except Exception as e:
    logging.error(f"Error loading sentences: {e}")

# 1. Load Vocabularies (using YOUR functions)
lulc_vocab = load_vocabulary_from_csv(LULC_VOCAB_PATH)
process_vocab = load_vocabulary_from_csv(PROCESS_VOCAB_PATH)

# 2. Setup SpaCy NLP Pipeline (using YOUR functions)
nlp_ner_pipeline = None
try:
    logging.info(f"Loading base SpaCy model '{BASE_SPACY_MODEL}' for NER...")
    nlp_ner_pipeline = spacy.load(BASE_SPACY_MODEL)
    
    # Generate patterns using YOUR function
    all_patterns_for_ruler = generate_ruler_patterns(nlp_ner_pipeline, lulc_vocab, process_vocab)
    
    # Add ruler using YOUR setup
    ruler_pipe_name = "custom_entity_ruler"
    if ruler_pipe_name in nlp_ner_pipeline.pipe_names:
        nlp_ner_pipeline.remove_pipe(ruler_pipe_name)
    
    ruler_config = {"overwrite_ents": True}
    if "ner" in nlp_ner_pipeline.pipe_names:
        ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, before="ner")
    else:
        ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, last=True)
    
    if all_patterns_for_ruler:
        ruler.add_patterns(all_patterns_for_ruler)
        logging.info(f"Custom EntityRuler '{ruler_pipe_name}' configured with {len(all_patterns_for_ruler)} patterns.")
        
except Exception as e:
    logging.error(f"Error setting up pipeline: {e}")

# 3. Process Sentences (YOUR code)
extracted_entities_data_structured = []
if nlp_ner_pipeline and not sentences_to_process_df.empty:
    print(f"\n--- Extracting Entities from {len(sentences_to_process_df)} test sentences ---")
    for index, row in sentences_to_process_df.iterrows():
        article_id = row.get(ARTICLE_ID_COL_IN_SENT_CSV, f"Row_{index}")
        sentence_text = row.get(SENTENCE_TEXT_COL_IN_SENT_CSV, "")

        if not isinstance(sentence_text, str) or not sentence_text.strip():
            continue

        doc = nlp_ner_pipeline(sentence_text)
        entities_in_sentence = []
        if doc.ents:
            for ent in doc.ents:
                normalized_ent_label = normalize_label(ent.label_)
                if 'TARGET_BASE_LABELS' in globals() and normalized_ent_label in TARGET_BASE_LABELS:
                    entities_in_sentence.append({
                        'text': ent.text,
                        'label': normalized_ent_label,
                        'start_char': ent.start_char,
                        'end_char': ent.end_char
                    })
        
        extracted_entities_data_structured.append({
             'article_id': article_id,
             'original_sentence': sentence_text,
             'entities': entities_in_sentence
         })

# 4. Save results (YOUR code)
if extracted_entities_data_structured:
    try:
        with open(OUTPUT_NER_JSON_PATH, 'w', encoding='utf-8') as f_json:
            json.dump(extracted_entities_data_structured, f_json, ensure_ascii=False, indent=4)
        print(f"\n✓ Saved results to: {OUTPUT_NER_JSON_PATH}")
    except Exception as e:
        print(f"Error saving: {e}")

# ========== END OF YOUR CODE ==========

# Step 5: Analyze the results
print("\n4. Analyzing extraction results...")
print("="*60)

if os.path.exists(OUTPUT_NER_JSON_PATH):
    with open(OUTPUT_NER_JSON_PATH, 'r') as f:
        results = json.load(f)
    
    # Show detailed results
    for entry in results:
        print(f"\nSentence: {entry['original_sentence']}")
        print("Extracted entities:")
        
        for ent in entry['entities']:
            word_count = len(ent['text'].split())
            multi_indicator = "**MULTI-WORD**" if word_count > 1 else "single"
            print(f"  - '{ent['text']}' → {ent['label']} ({multi_indicator})")
    
    # Summary statistics
    print("\n" + "="*60)
    print("SUMMARY:")
    total_entities = sum(len(e['entities']) for e in results)
    multi_word_count = sum(1 for e in results for ent in e['entities'] if len(ent['text'].split()) > 1)
    
    print(f"Total entities extracted: {total_entities}")
    print(f"Multi-word entities: {multi_word_count}")
    print(f"Single-word entities: {total_entities - multi_word_count}")
    
    # List all unique multi-word entities
    print("\nAll unique multi-word entities found:")
    multi_word_entities = set()
    for entry in results:
        for ent in entry['entities']:
            if len(ent['text'].split()) > 1:
                multi_word_entities.add((ent['text'], ent['label']))
    
    for text, label in sorted(multi_word_entities):
        print(f"  - '{text}' ({label})")

# Cleanup test files (optional)
print("\n5. Test complete! Test files created:")
print("  - test_LULC.csv")
print("  - test_LCprocess.csv") 
print("  - test_sentences.csv")
print("  - test_output_entities.json")
print("\nYou can delete these files after reviewing the results.")

=== TESTING YOUR ACTUAL CODE ===

1. Creating test vocabulary files...
   ✓ Created test_LULC.csv
   ✓ Created test_LCprocess.csv
   ✓ Created test_sentences.csv

2. Updating configuration variables...
   LULC_VOCAB_PATH = 'test_LULC.csv'
   PROCESS_VOCAB_PATH = 'test_LCprocess.csv'
   SENTENCES_CSV_PATH = 'test_sentences.csv'

3. Running YOUR entity extraction code...


NameError: name 'load_vocabulary_from_csv' is not defined

In [10]:
import pandas as pd
import re

# File path
CSV_FILE_PATH = 'annotated_corpus_for_dataverse.csv'

# Columns
ID_COLUMN = 'id_segment'
TEXT_COLUMN = 'text_segment'

# Define your preprocessing function
def preprocess_text(text):
    if pd.isnull(text):
        return ''
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove text in square brackets
    text = re.sub(r'\[.*?\]', '', text)
    # Remove special characters / punctuation (optional, customize as needed)
    text = re.sub(r'[^a-zA-Z0-9\s%.,]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Optional: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Read CSV
df = pd.read_csv(CSV_FILE_PATH)

# Apply preprocessing to the text column
df[TEXT_COLUMN] = df[TEXT_COLUMN].apply(preprocess_text)

# Save the cleaned DataFrame back to CSV
df.to_csv('preprocessed_' + CSV_FILE_PATH, index=False)

In [1]:
# Cell 10: NER on Sentences Loaded from CSV with Preprocessing (Saving as JSON)

import spacy
from spacy.tokens import Doc
import pandas as pd
import logging
import json
import re  # Added for preprocessing regex

# --- Configuration for this Cell ---
# Update this to your CSV file path
CSV_FILE_PATH = 'annotated_corpus_for_dataverse.csv'

# Column names in your CSV
ID_COLUMN = 'id_segment'  # Changed from 'article_id'
TEXT_COLUMN = 'text_segment'  # Changed from 'sentence'
RELEVANCE_COLUMN = 'relevance_label'

# Output JSON file
OUTPUT_NER_JSON_PATH = "extracted_ALL_entities_structured.json"

# --- Text Preprocessing Function ---
def preprocess_text(text):
    """Specialized preprocessing to handle TEI/XML artifacts"""
    if pd.isnull(text):
        return ''
    
    # Convert to string if not already
    if not isinstance(text, str):
        text = str(text)
    
    # --- FIRST PASS: Handle specific TEI patterns ---
    
    # Remove #text': pattern (common in TEI documents)
    text = re.sub(r'#text\':', '', text)
    text = re.sub(r'#text":', '', text)
    text = re.sub(r'"#text":', '', text)
    
    # Remove @xmlns patterns
    text = re.sub(r'@xmlns[^,]+,', '', text)
    text = re.sub(r'@xmlns[^\']+\'', '', text)
    text = re.sub(r'@xmlns[^"]+\"', '', text)
    
    # Remove any other TEI attributes
    text = re.sub(r'@[a-zA-Z0-9:_-]+[=][^\s,]+[,]?', '', text)
    
    # --- SECOND PASS: General XML cleanup ---
    
    # Remove XML tags
    text = re.sub(r'</?[^>]+>', '', text)
    
    # Remove any lingering single # or @ characters
    text = re.sub(r'(?<![a-zA-Z0-9])#(?![a-zA-Z0-9])', '', text)
    text = re.sub(r'(?<![a-zA-Z0-9])@(?![a-zA-Z0-9])', '', text)
    
    # --- THIRD PASS: Clean up quotes and other artifacts ---
    
    # Fix repeated quotes (common after removing TEI markers)
    text = re.sub(r'[\']{2,}', '\'', text)
    text = re.sub(r'[\"]{2,}', '\"', text)
    
    # Remove text in square brackets
    text = re.sub(r'\[.*?\]', '', text)
    
    # --- FINAL PASS: Standard text cleaning ---
    
    # Remove special characters except basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s%.,;:\'\"-]', ' ', text)
    
    # Fix spacing issues
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Final check for any remaining # or @ at the start of the text
    if text.startswith('#') or text.startswith('@'):
        text = text[1:]
    
    return text

# Helper function for label normalization
def normalize_label(label_str):
    if label_str in ['GPE', 'NORP']:
        return 'LOC'
    return label_str

# --- 0. Load, Preprocess, and Filter Data from CSV ---
print("🔍 Loading and preprocessing data...")
sentences_to_process_df = pd.DataFrame()
try:
    logging.info(f"Loading data from: {CSV_FILE_PATH}")
    df = pd.read_csv(CSV_FILE_PATH)
    
    # Apply preprocessing to the text column
    print("✨ Preprocessing text...")
    df[TEXT_COLUMN] = df[TEXT_COLUMN].apply(preprocess_text)
    print(f"✅ Preprocessing completed for {len(df)} text segments")
    
    # Filter for rows where relevance_label is not 4
    sentences_to_process_df = df[df[RELEVANCE_COLUMN] != 4].copy()
    print(f"✅ Filtered to {len(sentences_to_process_df)} rows with relevance_label != 4")
    
    # Verify required columns exist
    if ID_COLUMN not in sentences_to_process_df.columns or TEXT_COLUMN not in sentences_to_process_df.columns:
        raise ValueError(f"CSV must contain columns '{ID_COLUMN}' and '{TEXT_COLUMN}'")
    
    logging.info(f"Successfully loaded and preprocessed {len(sentences_to_process_df)} filtered rows from {CSV_FILE_PATH}.")
    
    # Optionally save preprocessed data
    preprocessed_path = 'preprocessed_' + CSV_FILE_PATH
    sentences_to_process_df.to_csv(preprocessed_path, index=False)
    print(f"💾 Saved preprocessed data to {preprocessed_path}")
    
except FileNotFoundError:
    logging.error(f"ERROR: CSV file not found at {CSV_FILE_PATH}.")
except ValueError as ve:
    logging.error(f"ERROR: {ve}")
except Exception as e:
    logging.error(f"An unexpected error occurred loading/preprocessing CSV: {e}")

# --- 1. Load Vocabularies ---
lulc_vocab = load_vocabulary_from_csv(LULC_VOCAB_PATH) if 'LULC_VOCAB_PATH' in globals() and LULC_VOCAB_PATH else set()
process_vocab = load_vocabulary_from_csv(PROCESS_VOCAB_PATH) if 'PROCESS_VOCAB_PATH' in globals() and PROCESS_VOCAB_PATH else set()

# --- 2. Setup SpaCy NLP Pipeline ---
nlp_ner_pipeline = None
if 'generate_ruler_patterns' in globals() and 'BASE_SPACY_MODEL' in globals():
    try:
        logging.info(f"Loading base SpaCy model '{BASE_SPACY_MODEL}' for NER...")
        nlp_ner_pipeline = spacy.load(BASE_SPACY_MODEL)
        logging.info(f"SpaCy model '{BASE_SPACY_MODEL}' loaded. Default pipes: {nlp_ner_pipeline.pipe_names}")
        logging.info("Generating NER patterns...")
        all_patterns_for_ruler = generate_ruler_patterns(nlp_ner_pipeline, lulc_vocab, process_vocab)

        ruler_pipe_name = "custom_entity_ruler"
        if ruler_pipe_name in nlp_ner_pipeline.pipe_names:
            nlp_ner_pipeline.remove_pipe(ruler_pipe_name)
        
        ruler_config = {"overwrite_ents": True}
        if "ner" in nlp_ner_pipeline.pipe_names:
            ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, before="ner")
        else:
            ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, last=True)
        
        if all_patterns_for_ruler:
            ruler.add_patterns(all_patterns_for_ruler)
            logging.info(f"Custom EntityRuler '{ruler_pipe_name}' configured with {len(all_patterns_for_ruler)} patterns.")
        else:
            logging.warning(f"Custom EntityRuler '{ruler_pipe_name}' added, but no patterns generated.")
        logging.info(f"Final NER pipeline: {nlp_ner_pipeline.pipe_names}")

    except Exception as e:
        logging.error(f"FATAL: Failed to set up NLP pipeline for NER: {e}", exc_info=True)
        nlp_ner_pipeline = None
else:
    logging.error("Required globals 'generate_ruler_patterns' or 'BASE_SPACY_MODEL' not found.")
    nlp_ner_pipeline = None

# --- 3. Process Text Segments and Extract Entities ---
extracted_entities_data_structured = []
all_entities = set()  # To collect all unique entities as in your original code

if nlp_ner_pipeline and not sentences_to_process_df.empty:
    print(f"\n📊 Processing texts and extracting entities from {len(sentences_to_process_df)} filtered rows...")
    
    for index, row in sentences_to_process_df.iterrows():
        text = row[TEXT_COLUMN]
        article_id = row[ID_COLUMN]
        
        print(f"\nProcessing ID: {article_id}")
        print(f"Text preview: {text[:100]}...")

        if not isinstance(text, str) or not text.strip():
            continue

        # Extract entities using the NER pipeline
        doc = nlp_ner_pipeline(text)
        entities_in_text = []
        
        if doc.ents:
            for ent in doc.ents:
                normalized_ent_label = normalize_label(ent.label_)
                if 'TARGET_BASE_LABELS' in globals() and normalized_ent_label in TARGET_BASE_LABELS:
                    entity_info = {
                        'text': ent.text,
                        'label': normalized_ent_label,
                        'start_char': ent.start_char,
                        'end_char': ent.end_char
                    }
                    entities_in_text.append(entity_info)
                    # Add to all_entities set (for compatibility with your original code)
                    all_entities.add((ent.text, normalized_ent_label))
        
        # Append entry even if no entities are found
        extracted_entities_data_structured.append({
            'article_id': article_id,
            'original_text': text,
            'entities': entities_in_text
        })
    
    print("\n--- Finished Entity Extraction ---")
    print(f"Total unique entities found: {len(all_entities)}")
    
elif sentences_to_process_df.empty:
    print("Skipping entity extraction: No rows with relevance_label != 4 found in the CSV file.")
elif not nlp_ner_pipeline:
    print("Skipping entity extraction: NER pipeline not set up.")

# --- 4. Display Sample Results and Save as JSON File ---
if extracted_entities_data_structured:
    print(f"\n--- Extracted Entities Data (showing first 3 entries) ---")
    for i, entry in enumerate(extracted_entities_data_structured[:3]):
        print(f"\nArticle ID: {entry['article_id']}")
        print(f"Text: {entry['original_text'][:200]}...")  # Show first 200 chars
        print("Entities:")
        if entry['entities']:
            for entity in entry['entities']:
                print(f"  - Text: '{entity['text']}', Label: {entity['label']}, Start: {entity['start_char']}, End: {entity['end_char']}")
        else:
            print("  (No entities extracted for this text based on TARGET_BASE_LABELS)")
    
    # --- SAVE AS A SINGLE JSON FILE ---
    try:
        with open(OUTPUT_NER_JSON_PATH, 'w', encoding='utf-8') as f_json:
            json.dump(extracted_entities_data_structured, f_json, ensure_ascii=False, indent=4)
        print(f"\n--- Successfully saved structured NER data to: {OUTPUT_NER_JSON_PATH} (single JSON file) ---")
        print(f"Total records saved: {len(extracted_entities_data_structured)}")
    except Exception as e_json_save:
        print(f"\nError saving structured NER data to JSON: {e_json_save}")

else:
    print("\nNo data from entity extraction to display or save.")
    # Create an empty JSON file if no data is extracted
    try:
        with open(OUTPUT_NER_JSON_PATH, 'w', encoding='utf-8') as f_json:
            json.dump([], f_json, ensure_ascii=False, indent=4)  # Save an empty list
        print(f"\nCreated an empty JSON file at: {OUTPUT_NER_JSON_PATH}")
    except Exception as e_json_create_empty:
        print(f"\nError creating empty JSON file: {e_json_create_empty}")

# Print summary statistics
if all_entities:
    print(f"\n📊 Summary Statistics:")
    print(f"- Total text segments processed: {len(extracted_entities_data_structured)}")
    print(f"- Total unique entities found: {len(all_entities)}")
    print(f"- Text segments with entities: {sum(1 for entry in extracted_entities_data_structured if entry['entities'])}")
    print(f"- Text segments without entities: {sum(1 for entry in extracted_entities_data_structured if not entry['entities'])}")

ERROR:root:Required globals 'generate_ruler_patterns' or 'BASE_SPACY_MODEL' not found.


🔍 Loading and preprocessing data...
✨ Preprocessing text...
✅ Preprocessing completed for 803 text segments
✅ Filtered to 803 rows with relevance_label != 4
💾 Saved preprocessed data to preprocessed_annotated_corpus_for_dataverse.csv
Skipping entity extraction: NER pipeline not set up.

No data from entity extraction to display or save.

Created an empty JSON file at: extracted_ALL_entities_structured.json


In [3]:
import spacy
from spacy.tokens import Doc
import pandas as pd
import logging
import json
import re
import os

# --- Configuration ---
CSV_FILE_PATH = 'annotated_corpus_for_dataverse.csv'
OUTPUT_NER_JSON_PATH = "extracted_ALL_entities_structured.json"

# Ensure these are defined (should come from your previous cells)
LULC_VOCAB_PATH = 'LULC.csv'  # Update with your actual path
PROCESS_VOCAB_PATH = 'LCprocess.csv'  # Update with your actual path
BASE_SPACY_MODEL = "en_core_web_sm"
TARGET_BASE_LABELS = [
    "LULC", "PROCESS", "CHANGE", "SURFACE_UNIT", "COORDINATES",
    "GPE", "LOC", "DATE", "PERCENT", "QUANTITY", "CARDINAL", "RESEARCH_TERM"
]

# --- TEI/XML Clean-up Function ---
def preprocess_text(text):
    """Specialized preprocessing to handle TEI/XML artifacts"""
    if pd.isnull(text):
        return ''
    if not isinstance(text, str):
        text = str(text)
    
    # TEI/XML cleanup
    text = re.sub(r'#text\':', '', text)
    text = re.sub(r'#text":', '', text)
    text = re.sub(r'"#text":', '', text)
    text = re.sub(r'@xmlns[^,]+,', '', text)
    text = re.sub(r'@xmlns[^\']+\'', '', text)
    text = re.sub(r'@xmlns[^"]+\"', '', text)
    text = re.sub(r'@[a-zA-Z0-9:_-]+[=][^\s,]+[,]?', '', text)
    text = re.sub(r'</?[^>]+>', '', text)
    text = re.sub(r'(?<![a-zA-Z0-9])#(?![a-zA-Z0-9])', '', text)
    text = re.sub(r'(?<![a-zA-Z0-9])@(?![a-zA-Z0-9])', '', text)
    text = re.sub(r'[\']{2,}', '\'', text)
    text = re.sub(r'[\"]{2,}', '\"', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s%.,;:\'\"-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    if text.startswith('#') or text.startswith('@'):
        text = text[1:]
    return text

def normalize_label(label_str):
    """Normalize entity labels"""
    if label_str in ['GPE', 'NORP']:
        return 'LOC'
    return label_str

# --- Load and Preprocess the CSV file ---
sentences_to_process_df = pd.DataFrame()
try:
    print(f"Loading data from: {CSV_FILE_PATH}")
    df = pd.read_csv(CSV_FILE_PATH)
    
    print(f"CSV contains {len(df)} rows with columns:")
    print(list(df.columns))
    
    # Auto-detect column names
    id_candidates = [col for col in df.columns if 'id' in col.lower()]
    ARTICLE_ID_COL = id_candidates[0] if id_candidates else df.columns[0]
    print(f"Using {ARTICLE_ID_COL} as the article ID column")
    
    text_candidates = [col for col in df.columns if 
                      any(term in col.lower() for term in ['text', 'sent', 'content'])]
    SENTENCE_TEXT_COL = text_candidates[0] if text_candidates else None
    
    if not SENTENCE_TEXT_COL:
        # Find column with longest text content
        sample_row = df.iloc[0]
        text_lengths = {col: len(str(val)) for col, val in sample_row.items()}
        SENTENCE_TEXT_COL = max(text_lengths, key=text_lengths.get)
    
    print(f"Using {SENTENCE_TEXT_COL} as the text column")
    
    # Find relevance column
    relevance_candidates = [col for col in df.columns if 
                           any(term in col.lower() for term in ['rel', 'label', 'class'])]
    RELEVANCE_COLUMN = relevance_candidates[0] if relevance_candidates else None
    
    if RELEVANCE_COLUMN:
        print(f"Using {RELEVANCE_COLUMN} as the relevance column")
    
    # Preprocess text
    print("\n✨ Preprocessing text...")
    df[SENTENCE_TEXT_COL] = df[SENTENCE_TEXT_COL].apply(preprocess_text)
    print(f"✅ Preprocessing completed for {len(df)} text segments")
    
    # Filter if relevance column exists
    if RELEVANCE_COLUMN and RELEVANCE_COLUMN in df.columns:
        unique_values = df[RELEVANCE_COLUMN].unique()
        print(f"Unique values in {RELEVANCE_COLUMN}: {sorted(unique_values)}")
        
        if 4 in unique_values:
            sentences_to_process_df = df[df[RELEVANCE_COLUMN] != 4].copy()
            print(f"✅ Filtered to {len(sentences_to_process_df)} rows with {RELEVANCE_COLUMN} != 4")
        else:
            sentences_to_process_df = df.copy()
            print(f"✅ No value 4 found, using all {len(df)} rows")
    else:
        sentences_to_process_df = df.copy()
        print(f"✅ Using all {len(sentences_to_process_df)} rows")
    
    if sentences_to_process_df.empty:
        raise ValueError("After filtering, the dataframe is empty.")
    
    print(f"Final dataframe contains {len(sentences_to_process_df)} rows ready for processing")
        
except FileNotFoundError:
    print(f"ERROR: File not found at {CSV_FILE_PATH}")
    sentences_to_process_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    import traceback
    traceback.print_exc()
    sentences_to_process_df = pd.DataFrame()

# --- Load Vocabularies (if functions are available) ---
lulc_vocab = set()
process_vocab = set()

if 'load_vocabulary_from_csv' in globals():
    try:
        lulc_vocab = load_vocabulary_from_csv(LULC_VOCAB_PATH) if os.path.exists(LULC_VOCAB_PATH) else set()
        process_vocab = load_vocabulary_from_csv(PROCESS_VOCAB_PATH) if os.path.exists(PROCESS_VOCAB_PATH) else set()
        print(f"Loaded LULC vocab: {len(lulc_vocab)} terms")
        print(f"Loaded PROCESS vocab: {len(process_vocab)} terms")
    except Exception as e:
        print(f"Error loading vocabularies: {e}")
else:
    print("Vocabulary loading functions not available - using base SpaCy NER only")

# --- Setup SpaCy NLP Pipeline ---
nlp_ner_pipeline = None
try:
    print(f"Loading SpaCy model '{BASE_SPACY_MODEL}'...")
    nlp_ner_pipeline = spacy.load(BASE_SPACY_MODEL)
    print(f"SpaCy model loaded. Default pipes: {nlp_ner_pipeline.pipe_names}")
    
    # Add custom patterns if functions are available
    if 'generate_ruler_patterns' in globals() and (lulc_vocab or process_vocab):
        try:
            print("Generating custom NER patterns...")
            all_patterns = generate_ruler_patterns(nlp_ner_pipeline, lulc_vocab, process_vocab)
            
            ruler_pipe_name = "custom_entity_ruler"
            if ruler_pipe_name in nlp_ner_pipeline.pipe_names:
                nlp_ner_pipeline.remove_pipe(ruler_pipe_name)
            
            ruler_config = {"overwrite_ents": True}
            if "ner" in nlp_ner_pipeline.pipe_names:
                ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, 
                                                config=ruler_config, before="ner")
            else:
                ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, 
                                                config=ruler_config, last=True)
            
            if all_patterns:
                ruler.add_patterns(all_patterns)
                print(f"Custom EntityRuler configured with {len(all_patterns)} patterns.")
            
            print(f"Final pipeline: {nlp_ner_pipeline.pipe_names}")
        except Exception as e:
            print(f"Error setting up custom patterns: {e}")
            print("Continuing with base SpaCy NER only...")
    else:
        print("Using base SpaCy NER only (no custom vocabularies or pattern generation)")
        
except Exception as e:
    print(f"FATAL: Failed to load SpaCy model: {e}")
    nlp_ner_pipeline = None

# --- Process Sentences and Extract Entities ---
extracted_entities_data = []

if nlp_ner_pipeline and not sentences_to_process_df.empty:
    print(f"\n--- Extracting Entities from {len(sentences_to_process_df)} Sentences ---")
    
    for index, row in sentences_to_process_df.iterrows():
        article_id = row.get(ARTICLE_ID_COL, f"Row_{index}")
        sentence_text = row.get(SENTENCE_TEXT_COL, "")

        if not isinstance(sentence_text, str) or not sentence_text.strip():
            continue

        # Process with NER pipeline
        doc = nlp_ner_pipeline(sentence_text)
        entities_in_sentence = []
        
        if doc.ents:
            for ent in doc.ents:
                normalized_label = normalize_label(ent.label_)
                if normalized_label in TARGET_BASE_LABELS:
                    entities_in_sentence.append({
                        'text': ent.text,
                        'label': normalized_label,
                        'start_char': ent.start_char,
                        'end_char': ent.end_char
                    })

        # Add entry for this sentence
        extracted_entities_data.append({
            'article_id': article_id,
            'original_sentence': sentence_text,
            'entities': entities_in_sentence
        })
    
    print("--- Finished Entity Extraction ---")
else:
    if sentences_to_process_df.empty:
        print("No sentences loaded from CSV file.")
    if not nlp_ner_pipeline:
        print("NER pipeline not set up.")

# --- Display Results and Save JSON ---
if extracted_entities_data:
    print(f"\n--- Sample Results (first 3 entries) ---")
    for i, entry in enumerate(extracted_entities_data[:3]):
        print(f"\nArticle ID: {entry['article_id']}")
        print(f"Sentence: {entry['original_sentence'][:100]}...")
        print("Entities:")
        if entry['entities']:
            for entity in entry['entities']:
                print(f"  - Text: '{entity['text']}', Label: {entity['label']}")
        else:
            print("  (No entities found)")
    
    # Save as JSON
    try:
        with open(OUTPUT_NER_JSON_PATH, 'w', encoding='utf-8') as f:
            json.dump(extracted_entities_data, f, ensure_ascii=False, indent=4)
        print(f"\n--- Successfully saved NER data to: {OUTPUT_NER_JSON_PATH} ---")
        
        # Summary statistics
        total_entities = sum(len(entry['entities']) for entry in extracted_entities_data)
        sentences_with_entities = sum(1 for entry in extracted_entities_data if entry['entities'])
        print(f"Total sentences processed: {len(extracted_entities_data)}")
        print(f"Sentences with entities: {sentences_with_entities}")
        print(f"Total entities extracted: {total_entities}")
        
    except Exception as e:
        print(f"Error saving JSON file: {e}")
else:
    print("No data extracted to save.")

Loading data from: annotated_corpus_for_dataverse.csv
CSV contains 803 rows with columns:
['Unnamed: 0', 'id_segment', 'text_segment', 'relevance_label', 'relevance_type_norm']
Using id_segment as the article ID column
Using text_segment as the text column
Using relevance_label as the relevance column

✨ Preprocessing text...
✅ Preprocessing completed for 803 text segments
Unique values in relevance_label: [0, 1, 2]
✅ No value 4 found, using all 803 rows
Final dataframe contains 803 rows ready for processing
Vocabulary loading functions not available - using base SpaCy NER only
Loading SpaCy model 'en_core_web_sm'...
SpaCy model loaded. Default pipes: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Using base SpaCy NER only (no custom vocabularies or pattern generation)

--- Extracting Entities from 803 Sentences ---
--- Finished Entity Extraction ---

--- Sample Results (first 3 entries) ---

Article ID: 1-s2.0-S0301479717300713-main_226b
Sentence: ' Lambin et 

In [4]:
# Add this debugging code after generating patterns
print(f"LULC vocabulary size: {len(lulc_vocab)}")
lulc_patterns = [p for p in all_patterns_for_ruler if p.get('label') == 'LULC']
print(f"Generated LULC patterns: {len(lulc_patterns)}")
print("Sample LULC patterns:", lulc_patterns[:5])

# After processing, check if LULC entities are being recognized
lulc_entities_found = 0
for entry in extracted_entities_data_structured:
    lulc_entities_found += sum(1 for e in entry['entities'] if e['label'] == 'LULC')
print(f"Total LULC entities extracted: {lulc_entities_found}")

# Validate the pattern generation by testing on known LULC terms
test_lulc_texts = list(lulc_vocab)[:10]  # Test first 10 terms
print("\nTesting LULC term recognition:")
for test_text in test_lulc_texts:
    doc = nlp_ner_pipeline(test_text)
    if doc.ents:
        print(f"'{test_text}' recognized as: {[(ent.text, ent.label_) for ent in doc.ents]}")
    else:
        print(f"'{test_text}' NOT recognized as any entity")

LULC vocabulary size: 0


NameError: name 'all_patterns_for_ruler' is not defined

In [9]:
# === Single Cell: NER Pipeline with Manual Vocabulary Lists ===

import spacy
import logging

# === Target Labels Filter ===
BASE_LABELS = [
    'CHANGE',       # transformation words (increased, decreased)
    'LOC',          # location names
    'LULC',         # Land Use/Land Cover (THIS IS YOUR MAIN FOCUS)
    'DATE',         # temporal references
    'PERCENT',      # percentage values
    'CARDINAL',     # numeric values
    'COORDINATES',  # geographic coordinates
    'SURFACE_UNIT', # area measurements
    'PROCESS',      # environmental processes
    'QUANTITY'      # other quantities
]

# === Manual Vocabulary Lists (Edit these as needed) ===
MANUAL_LOC_VOCAB_LIST =  [
    "Sahel areas", 
    "Miombo",  
    "Congo Basin Rainforest",
    "Savanna",
    "African Mangroves region",
    "African Mangroves",
    "Sahel",
    "Fynbos",
    "Sahara Desert",
    "Kalahari Desert",
    "Namib Desert",
    "Amazon Rainforest",
    "Cerrado",
    "Atlantic Forest",
    "Pampas",
    "Caatinga",
    "Pantanal",
    "Gran Chaco",
    "Mediterranean Forest",
    "Maquis",
    "Garrigue",
    "Heathlands",
    "Moorlands",
    "Pontic Steppe",
    "Taiga (Europe)",
    "Eurasian Steppe",
    "Sundarbans Mangroves",
    "Borneo Rainforest",
    "Monsoon Dry Forest",
    "Gobi Desert",
    "Karakum Desert",
    "Kyzylkum Desert",
    "Arabian Desert",
    "Taiga (Siberia)",
    "Tallgrass Prairie",
    "Mixed-grass Prairie",
    "Shortgrass Prairie",
    "Eastern Deciduous Forest",
    "Chaparral",
    "Sonoran Desert",
    "Chihuahuan Desert",
    "Mojave Desert",
    "Great Basin Desert",
    "Everglades",
    "Taiga (Canada)",
    "Australian Tropical Savanna",
    "Mallee",
    "Mulga",
    "Spinifex Desert",
    "New Guinea Rainforest",
    "New Zealand Temperate Forest",
    "Arctic Tundra",
    "Taiga",
    "Polar Desert",
    "Ice Cap"
]

# Load vocabularies from CSV files (if they exist) or use empty sets
manual_loc_vocab = set([loc.lower().strip() for loc in MANUAL_LOC_VOCAB_LIST])
manual_lulc_vocab = load_vocabulary_from_csv(LULC_VOCAB_PATH) if 'LULC_VOCAB_PATH' in globals() and LULC_VOCAB_PATH else set()
manual_process_vocab = load_vocabulary_from_csv(PROCESS_VOCAB_PATH) if 'PROCESS_VOCAB_PATH' in globals() and PROCESS_VOCAB_PATH else set()

# === Test Sentences ===
sentences_with_ids = [
    ("1", "Agricultural expansion occurred in the Miombo area."),
    ("2", "Agricultural expansion occurred in the Congo Basin Rainforest area."),
    ("3", "Deforestation has affected the Savanna ecosystem substantially."),
    ("4", "Land cover analysis in the Sahel areas showed decreasing forest cover."),
    ("5", "The Fynbos region has experienced significant land use changes."),
    ("6", "Land cover analysis in the Sahara Desert areas showed decreasing forest cover."),
    ("7", "Agricultural expansion occurred in the Kalahari Desert area."),
    ("8", "Remote sensing data showed changes in Namib Desert vegetation."),
    ("9", "The African Mangroves region has experienced significant land use changes."),
    ("10", "Remote sensing data showed changes in Amazon Rainforest vegetation."),
    ("11", "Remote sensing data showed changes in Cerrado vegetation."),
    ("12", "The Atlantic Forest region has experienced significant land use changes."),
    ("13", "Remote sensing data showed changes in Pampas vegetation."),
    ("14", "Remote sensing data showed changes in Caatinga vegetation."),
    ("15", "Remote sensing data showed changes in Pantanal vegetation."),
    ("16", "Land cover analysis in the Gran Chaco areas showed decreasing forest cover."),
    ("17", "Remote sensing data showed changes in Mediterranean Forest vegetation."),
    ("18", "The Maquis region has experienced significant land use changes."),
    ("19", "Remote sensing data showed changes in Garrigue vegetation."),
    ("20", "Remote sensing data showed changes in Heathlands vegetation."),
    ("21", "The Moorlands region has experienced significant land use changes."),
    ("22", "Remote sensing data showed changes in Pontic Steppe vegetation."),
    ("23", "Remote sensing data showed changes in Taiga (Europe) vegetation."),
    ("24", "Remote sensing data showed changes in Eurasian Steppe vegetation."),
    ("25", "Remote sensing data showed changes in Sundarbans Mangroves vegetation."),
    ("26", "Remote sensing data showed changes in Borneo Rainforest vegetation."),
    ("27", "Remote sensing data showed changes in Monsoon Dry Forest vegetation."),
    ("28", "Remote sensing data showed changes in Gobi Desert vegetation."),
    ("29", "Remote sensing data showed changes in Karakum Desert vegetation."),
    ("30", "Remote sensing data showed changes in Kyzylkum Desert vegetation."),
    ("31", "Remote sensing data showed changes in Arabian Desert vegetation."),
    ("32", "Remote sensing data showed changes in Taiga (Siberia) vegetation."),
    ("33", "Remote sensing data showed changes in Tallgrass Prairie vegetation."),
    ("34", "Remote sensing data showed changes in Mixed-grass Prairie vegetation."),
    ("35", "Remote sensing data showed changes in Shortgrass Prairie vegetation."),
    ("36", "Remote sensing data showed changes in Eastern Deciduous Forest vegetation."),
    ("37", "Remote sensing data showed changes in Chaparral vegetation."),
    ("38", "Remote sensing data showed changes in Sonoran Desert vegetation."),
    ("39", "Remote sensing data showed changes in Chihuahuan Desert vegetation."),
    ("40", "Remote sensing data showed changes in Mojave Desert vegetation."),
    ("41", "Remote sensing data showed changes in Great Basin Desert vegetation."),
    ("42", "Remote sensing data showed changes in Everglades vegetation."),
    ("43", "Remote sensing data showed changes in Taiga (Canada) vegetation."),
    ("44", "Remote sensing data showed changes in Australian Tropical Savanna vegetation."),
    ("45", "Remote sensing data showed changes in Mallee vegetation."),
    ("46", "Remote sensing data showed changes in Mulga vegetation."),
    ("47", "Remote sensing data showed changes in Spinifex Desert vegetation."),
    ("48", "Remote sensing data showed changes in New Guinea Rainforest vegetation."),
    ("49", "Remote sensing data showed changes in New Zealand Temperate Forest vegetation."),
    ("50", "Remote sensing data showed changes in Arctic Tundra vegetation."),
    ("51", "Remote sensing data showed changes in Taiga vegetation."),
    ("52", "Remote sensing data showed changes in Polar Desert vegetation."),
    ("53", "Remote sensing data showed changes in Ice Cap vegetation."),
    ("54", "I visited Paris and Berlin in the summer for forest conservation."),
    ("55", "New York urban development has impacted agricultural areas."),
    ("56", "Deforestation in the Amazon rainforest affects global climate."),
    ("57", "Tokyo has excellent wetland restoration projects."),
    ("58", "Agricultural expansion in California requires careful planning."),
    ("59", "This sentence has no relevant entities to extract.")
]

# === Helper Functions ===

def normalize_label(label):
    """Normalize SpaCy default labels to our custom ones and filter by BASE_LABELS"""
    if label in ['GPE', 'NORP']:
        return 'LOC'
    elif label in ['PERSON']:
        return None  # Filter out - not in BASE_LABELS
    elif label in ['ORG']:
        return None  # Filter out - not in BASE_LABELS
    elif label in ['DATE']:
        return 'DATE'
    elif label in ['PERCENT']:
        return 'PERCENT'
    elif label in ['CARDINAL']:
        return 'CARDINAL'
    elif label in ['QUANTITY']:
        return 'QUANTITY'
    elif label in BASE_LABELS:
        return label
    else:
        return None  # Filter out labels not in BASE_LABELS

def generate_simple_patterns(vocab_set, label):
    """Generate LOWER token patterns for vocabulary - improved to handle multi-word terms properly"""
    patterns = []
    
    # Create a temporary nlp object for tokenization
    temp_nlp = spacy.blank("en")
    
    for term in sorted(vocab_set):
        if term.strip():
            # Use SpaCy tokenizer to properly split the term
            doc = temp_nlp(term.lower())
            tokens = [token.text for token in doc if token.text.strip()]
            
            if tokens:
                pattern_tokens = [{"LOWER": token} for token in tokens]
                patterns.append({"label": label, "pattern": pattern_tokens})
                
    return patterns

def generate_extended_patterns(vocab_set, label):
    """Generate patterns for terms + common extensions like 'region', 'area', etc."""
    patterns = []
    temp_nlp = spacy.blank("en")
    
    # Common extensions for location names
    extensions = ["region", "area", "areas", "ecosystem", "zone", "zones", "vegetation"]
    
    for term in sorted(vocab_set):
        if term.strip():
            # Base term pattern
            doc = temp_nlp(term.lower())
            base_tokens = [token.text for token in doc if token.text.strip()]
            
            if base_tokens:
                # Add base pattern
                base_pattern = [{"LOWER": token} for token in base_tokens]
                patterns.append({"label": label, "pattern": base_pattern})
                
                # Add patterns with extensions
                for ext in extensions:
                    extended_pattern = base_pattern + [{"LOWER": ext}]
                    patterns.append({"label": label, "pattern": extended_pattern})
                    
    return patterns

# === Build All Patterns ===
all_patterns = []

# Add patterns for each vocabulary type - using extended patterns for locations
all_patterns.extend(generate_extended_patterns(manual_loc_vocab, "LOC"))
all_patterns.extend(generate_simple_patterns(manual_lulc_vocab, "LULC"))
all_patterns.extend(generate_simple_patterns(manual_process_vocab, "PROCESS"))

# Add CHANGE patterns
change_terms = ["increase", "decrease", "loss", "gain", "expansion", "reduction", "growth", "change", "decline", "improve", "transform"]
change_patterns = [{"label": "CHANGE", "pattern": [{"LOWER": term}]} for term in change_terms]
all_patterns.extend(change_patterns)

# Add some coordinate patterns (simplified)
coordinate_patterns = [
    {"label": "COORDINATES", "pattern": [{"LIKE_NUM": True}, {"LOWER": "n"}, {"LIKE_NUM": True}, {"LOWER": "e"}]},
    {"label": "COORDINATES", "pattern": [{"LIKE_NUM": True}, {"TEXT": "°"}, {"LOWER": {"IN": ["n", "s"]}}, {"LIKE_NUM": True}, {"TEXT": "°"}, {"LOWER": {"IN": ["e", "w"]}}]},
]
all_patterns.extend(coordinate_patterns)

# Add surface unit patterns
surface_patterns = [
    {"label": "SURFACE_UNIT", "pattern": [{"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectare", "hectares"]}}]},
    {"label": "SURFACE_UNIT", "pattern": [{"LIKE_NUM": True}, {"LOWER": {"IN": ["acres", "acre"]}}]},
    {"label": "SURFACE_UNIT", "pattern": [{"LIKE_NUM": True}, {"LOWER": {"IN": ["km", "kilometers", "sq"]}}, {"LOWER": {"IN": ["km", "kilometers"]}, "OP": "?"}]},
]
all_patterns.extend(surface_patterns)

print(f"Generated {len(all_patterns)} total patterns")

# === Setup SpaCy Pipeline ===
try:
    # Load base model with default NER
    nlp = spacy.load("en_core_web_sm")
    print(f"Loaded SpaCy model. Default pipes: {nlp.pipe_names}")
    
    # Add custom EntityRuler BEFORE default NER so it takes priority
    if "custom_entity_ruler" in nlp.pipe_names:
        nlp.remove_pipe("custom_entity_ruler")
    
    # Add ruler BEFORE NER so custom patterns take priority
    ruler = nlp.add_pipe("entity_ruler", name="custom_entity_ruler", config={"overwrite_ents": True}, before="ner")
    ruler.add_patterns(all_patterns)
    
    print(f"Added EntityRuler with {len(all_patterns)} patterns")
    print(f"Final pipeline: {nlp.pipe_names}")
    
except Exception as e:
    print(f"Error setting up pipeline: {e}")
    # Fallback to blank model
    nlp = spacy.blank("en")
    ruler = nlp.add_pipe("entity_ruler", name="custom_entity_ruler")
    ruler.add_patterns(all_patterns)
    print("Using blank model with custom EntityRuler only")

# === Process Sentences and Extract Entities ===
print("\n" + "="*60)
print("EXTRACTING NAMED ENTITIES FROM SENTENCES")
print("="*60)
all_results = []
json_results = []

for i, (article_id, sentence) in enumerate(sentences_with_ids, 1):
    print(f"\n--- Sentence {i} ---")
    print(f"Article ID: {article_id}")
    print(f"Text: {sentence}")
    
    # Process with NLP pipeline
    doc = nlp(sentence)
    
    # Extract entities with filtering
    entities = []
    json_entities = []
    
    for ent in doc.ents:
        normalized_label = normalize_label(ent.label_)
        
        # Only keep entities with labels in BASE_LABELS
        if normalized_label and normalized_label in BASE_LABELS:
            entities.append({
                'text': ent.text,
                'label': normalized_label,
                'start': ent.start_char,
                'end': ent.end_char,
                'original_label': ent.label_
            })
            
            # Format for JSON output
            json_entities.append({
                "text": ent.text,
                "label": normalized_label,
                "start_char": ent.start_char,
                "end_char": ent.end_char
            })
    
    # Store results for display
    result = {
        'sentence_id': i,
        'article_id': article_id,
        'sentence': sentence,
        'entities': entities,
        'entity_count': len(entities)
    }
    all_results.append(result)
    
    # Store results for JSON export
    json_result = {
        "article_id": article_id,
        "original_sentence": sentence,
        "entities": json_entities
    }
    json_results.append(json_result)
    
    # Print entities
    if entities:
        print(f"Found {len(entities)} entities:")
        for ent in entities:
            print(f"  • '{ent['text']}' → {ent['label']} (orig: {ent['original_label']}) [{ent['start']}:{ent['end']}]")
    else:
        print("  No entities found.")

# === Save JSON Output ===
output_dir = "ner_results"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save individual JSON files for each sentence
for result in json_results:
    filename = f"{result['article_id']}.json"
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"Saved: {filepath}")

# Save combined JSON file
combined_filepath = os.path.join(output_dir, "all_ner_results.json")
with open(combined_filepath, 'w', encoding='utf-8') as f:
    json.dump(json_results, f, ensure_ascii=False, indent=2)

print(f"\nSaved combined results: {combined_filepath}")

# === Summary ===
print(f"\n" + "="*60)
print("SUMMARY")
print("="*60)

total_entities = sum(result['entity_count'] for result in all_results)
print(f"Processed {len(sentences_with_ids)} sentences")
print(f"Found {total_entities} total entities")

# Count by label
label_counts = {}
for result in all_results:
    for ent in result['entities']:
        label = ent['label']
        label_counts[label] = label_counts.get(label, 0) + 1

if label_counts:
    print("\nEntities by label:")
    for label, count in sorted(label_counts.items()):
        print(f"  {label}: {count}")

print(f"\nVocabulary sizes used:")
print(f"  LOC: {len(manual_loc_vocab)} terms")
print(f"  LULC: {len(manual_lulc_vocab)} terms") 
print(f"  PROCESS: {len(manual_process_vocab)} terms")

print(f"\nFiltered to BASE_LABELS: {BASE_LABELS}")
print(f"\nJSON files saved in: {output_dir}/")

# === Display sample JSON format ===
print(f"\n" + "="*60)
print("SAMPLE JSON OUTPUT")
print("="*60)
if json_results:
    sample_json = json_results[0]
    print(json.dumps(sample_json, indent=2, ensure_ascii=False))

2025-07-04 17:44:07,330 - INFO - Attempting to load vocabulary from: LULC.csv using term column: 'term'
2025-07-04 17:44:07,335 - INFO - Successfully loaded 172 unique terms from LULC.csv (column: 'term')
2025-07-04 17:44:07,337 - INFO - Attempting to load vocabulary from: LCprocess.csv using term column: 'term'
2025-07-04 17:44:07,340 - INFO - Successfully loaded 19 unique terms from LCprocess.csv (column: 'term')


Generated 647 total patterns
Loaded SpaCy model. Default pipes: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Added EntityRuler with 647 patterns
Final pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'custom_entity_ruler', 'ner']

EXTRACTING NAMED ENTITIES FROM SENTENCES

--- Sentence 1 ---
Article ID: 1
Text: Agricultural expansion occurred in the Miombo area.
Found 2 entities:
  • 'expansion' → CHANGE (orig: CHANGE) [13:22]
  • 'Miombo area' → LOC (orig: LOC) [39:50]

--- Sentence 2 ---
Article ID: 2
Text: Agricultural expansion occurred in the Congo Basin Rainforest area.
Found 2 entities:
  • 'expansion' → CHANGE (orig: CHANGE) [13:22]
  • 'Congo Basin Rainforest area' → LOC (orig: LOC) [39:66]

--- Sentence 3 ---
Article ID: 3
Text: Deforestation has affected the Savanna ecosystem substantially.
Found 2 entities:
  • 'Deforestation' → PROCESS (orig: PROCESS) [0:13]
  • 'Savanna ecosystem' → LOC (orig: LOC) [31:48]

--- Sentence 